# 01 · Prepare data

TinyShakespeare is downloaded from Karpathy's char-rnn repository and split contiguously 80/10/10. Each cut is moved forward to the next line break, so no line is split across sets. The vocabulary comes from the training split only, and any validation or test characters missing from it are reported.

In [ ]:
import json
import urllib.request
from pathlib import Path

import torch

DATA_URL = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
DATA_PATH = Path("../data/input.txt")
RESULTS_DIR = Path("../results")
DATA_PATH.parent.mkdir(exist_ok=True)
RESULTS_DIR.mkdir(exist_ok=True)

if not DATA_PATH.exists():
    urllib.request.urlretrieve(DATA_URL, DATA_PATH)

In [ ]:
text = DATA_PATH.read_text(encoding="utf-8")

def next_line_start(position):
    return text.index("\n", position) + 1

train_end = next_line_start(int(0.8 * len(text)))
val_end = next_line_start(int(0.9 * len(text)))
train_text, val_text, test_text = text[:train_end], text[train_end:val_end], text[val_end:]

vocab = sorted(set(train_text))
char_to_id = {char: i for i, char in enumerate(vocab)}
unseen_chars = {name: sorted(set(split) - set(vocab)) for name, split in [("val", val_text), ("test", test_text)]}

def encode(split_text):
    return torch.tensor([char_to_id[char] for char in split_text])

train_data, val_data, test_data = encode(train_text), encode(val_text), encode(test_text)

data_summary = {
    "total_chars": len(text),
    "split_chars": {"train": len(train_text), "val": len(val_text), "test": len(test_text)},
    "vocab_size": len(vocab),
    "unseen_chars": unseen_chars,
}
(RESULTS_DIR / "data_summary.json").write_text(json.dumps(data_summary, indent=2))
data_summary